### Define variables
Initial gb and fasta files from entire project should be first defined.

In [2]:
workdir = "plastomes/table2asn/cp"
cc_gb = "plastomes/table2asn/cc/fixed_table2asn/output.gbf"
cc_fna = "plastomes/table2asn/fasta_manual_description/Crepis_callicephala.fasta"

cp_gb = "fixed_Crepis_purpurea.gb"
cp_fna = "plastomes/final/gb2sequin_out/c_purpurea_final/03/Crepis_purpurea.fsa"

### Check CDS of genes with known RNA editing sites

In [3]:
from Bio import SeqIO, SeqRecord

# output psbL data
gb_file = cp_gb # file with fixed tRNAs
target_gene = "psbL"

with open(gb_file) as handle:
    for record in SeqIO.parse(handle, "gb"):
        for feature in record.features:
            if feature.type in ["CDS", "gene"]:
                if feature.qualifiers.get('gene')[0] == target_gene:
                    print(feature)
                    cds_seq = feature.extract(record.seq)
                    #print(feature.qualifiers.get('gene')[0], cds_seq[:20])
                    start = feature.location.start - 1
                    end = start + 3
                    codon = record.seq[start:end].reverse_complement()  # 0-based
                    #print(f"codon: {codon}")

type: gene
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']

type: CDS
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']
    Key: product, Value: ['photosystem II subunit L']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['TTQSNPNEQNVELNRTSLYWGLLLIFVLAVLFSNYFFN']



The _**psbL**_ has no canonical codon in coding sequence start position.
Exception should be added and transcription output should be modified by changing the first AA letter to M

In [4]:
# output ndhD data
target_gene = "ndhD"

aa_shift = 6
bp_shift = aa_shift * 3

for record in SeqIO.parse(gb_file, "gb"):
    for feature in record.features:
        if feature.type == "CDS":
            gene_name = feature.qualifiers.get("gene", [""])[0]
            if gene_name == target_gene:
                print(feature)
                
                start = int(feature.location.start)
                end = int(feature.location.end)
                strand = feature.location.strand
                na = record.seq[start : end]

                if strand == 1:
                    position_shifted = start + bp_shift
                    na_shifted = record.seq[start + bp_shift : end]
                    ncbi_coord_str = f"{position_shifted + 1}..{end}"
                elif strand == -1:
                    position_shifted = end - bp_shift
                    na_shifted = record.seq[start : end - bp_shift]
                    ncbi_coord_str = f"{start + 1}..{position_shifted}"
                else:
                    print("Strand orientation unknown.")
                    continue
                
                for na_seq in [na, na_shifted]:
                    # convert to 5'-3' before translation
                    coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                    aa_seq = coding_seq.translate(to_stop=False)
                    if na_seq == na_shifted:
                        print(f"Translated sequence first symbols (shifted {aa_shift} aa): {aa_seq[0:15]}")
                    else:
                        print(f"Translated sequence first symbols: {aa_seq[0:21]}")
                    

                print(f"\ngene strand: ({'+' if strand == 1 else '-'})")
                print(f"Original gene coordinates in Biopython are [{start}:{end}]")
                print(f"Original GenBank coordinates: {start + 1}..{end}")
                print(f"Adjusted (start for (+) or end for (-) strand) position after shifting by {aa_shift} AA in Biopython is {position_shifted}")
                print(f"New GenBank coordinates after {aa_shift} aa shift: {ncbi_coord_str}")

type: CDS
location: [112421:113942](-)
qualifiers:
    Key: gene, Value: ['ndhD']
    Key: product, Value: ['NADH dehydrogenase subunit D']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR']

Translated sequence first symbols: VYLVFTTNKFPWLTIIVVLPI
Translated sequence first symbols (shifted 6 aa): TNKFPWLTIIVVLPI

gene strand: (-)
Original gene coordinates in Biopython are [112421:113942]
Original GenBank coordinates: 112422..113942
Adjusted (start for (+) or end for (-

As we see, in Crepis purpurea, the _**ndhD**_ has 6 additional amino acids in the start of its CDS sequence, but C. callicephala has not. It may represent the annotation error in C. purpurea and should be fixed by adjusting the gene boundaries.

It also has non-canonical codon at the coding sequence start position. The first AA should be changed to M after adding exception.

So there are two issues:
1. Start position needs adjustment.
2. CDS translation needs modifying by changing the first AA letter to M.

### Fixing non-canonical start codon-related issues via code
#### Granular approach
As we need changing two genes with different combination of things to change, the most preferred is a granular step-by-step approach, when we:
1. Adjust _**ndhD**_ coordinates in `CDS` and `gene` features.
2. Fix start position issues in `CDS` features of _**ndhD**_ and _**psbL**_.

In [5]:
# 1. Adjust coordinates
from Bio import SeqIO
from Bio.SeqFeature import FeatureLocation
import os

# Configuration
input_gb = cp_gb
outdir = f"{workdir}/01_prot_ndhD"
output_gb = f"{outdir}/cp_ndhD-coord-fixed.gb"
target_gene = "ndhD"
aa_shift = 6
bp_shift = aa_shift * 3

os.makedirs(outdir, exist_ok=True)

# Load all records into memory
records = list(SeqIO.parse(input_gb, "gb"))

for record in records:
    for feature in record.features:
        # Safely check for target gene
        gene_name = feature.qualifiers.get("gene", [""])[0]
        if feature.type in ["gene", "CDS"] and gene_name == target_gene:
            old_start = int(feature.location.start)
            old_end = int(feature.location.end)
            strand = int(feature.location.strand)

            # Calculate new boundaries
            if strand == -1:
                # Reverse strand: downstream shift reduces the end coordinate
                new_start = old_start
                new_end = old_end - bp_shift
            elif strand == 1:
                # Forward strand: downstream shift increases the start coordinate
                new_start = old_start + bp_shift
                new_end = old_end
            else:
                continue

            # Validate that new length maintains open reading frame
            if (new_end - new_start) % 3 != 0:
                raise ValueError(f"New length for {target_gene} is not a multiple of 3.")

            # Update feature location (0-based, half-open)
            feature.location = FeatureLocation(new_start, new_end, strand)
            print(f"[{feature.type}] feature coords of {gene_name} for target gene {target_gene} was updated: {old_start}..{old_end} -> {new_start}..{new_end} (0-based)")

            # update feature translation length
            if feature.type == 'CDS':
                trans_old = feature.qualifiers.get('translation', "")[0]
                # convert to 5'-3' before translation
                coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                trans_new = coding_seq.translate(to_stop=True)
                feature.qualifiers['translation'] = [trans_new]
                print(f"old translation: {trans_old}\nnew translation: {trans_new}")

# Write corrected file
SeqIO.write(records, output_gb, "gb")
print(f"\nSaved corrected GenBank file to: {output_gb}")

[gene] feature coords of ndhD for target gene ndhD was updated: 112421..113942 -> 112421..113924 (0-based)
[CDS] feature coords of ndhD for target gene ndhD was updated: 112421..113942 -> 112421..113924 (0-based)
old translation: VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR
new translation: TNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTC

In [6]:
# check gene output
target_gene = "ndhD"

aa_shift = 6
bp_shift = aa_shift * 3
gb_old = cp_gb
gb_new = output_gb

for gb_file in [gb_old, gb_new]:
    for record in SeqIO.parse(gb_file, "gb"):
        print(f"______________________________________________\nGenBank file: {gb_file}")
        for feature in record.features:
            if feature.type == "CDS":
                gene_name = feature.qualifiers.get("gene", [""])[0]
                if gene_name == target_gene:
                    #print(feature)
                    
                    start = int(feature.location.start)
                    end = int(feature.location.end)
                    strand = feature.location.strand
                    na = record.seq[start : end]

                    if strand == 1:
                        position_shifted = start + bp_shift
                        na_shifted = record.seq[start + bp_shift : end]
                        ncbi_coord_str = f"{position_shifted + 1}..{end}"
                    elif strand == -1:
                        position_shifted = end - bp_shift
                        na_shifted = record.seq[start : end - bp_shift]
                        ncbi_coord_str = f"{start + 1}..{position_shifted}"
                    else:
                        print("Strand orientation unknown.")
                        continue
                    
                    for na_seq in [na]:
                        # convert to 5'-3' before translation
                        coding_seq = na_seq.reverse_complement() if strand == -1 else na_seq
                        aa_seq = coding_seq.translate(to_stop=False)
                        print(f"Translated sequence first symbols: {aa_seq[0:21]}")
                        

                    print(f"\ngene strand: ({'+' if strand == 1 else '-'})")
                    print(f"Original coordinates in Biopython: [{start}:{end}]")
                    print(f"Original coordinates in GenBank: {start + 1}..{end}")
                    print(f"CDS feature translation: {feature.qualifiers.get('translation', "")}")

______________________________________________
GenBank file: fixed_Crepis_purpurea.gb
Translated sequence first symbols: VYLVFTTNKFPWLTIIVVLPI

gene strand: (-)
Original coordinates in Biopython: [112421:113942]
Original coordinates in GenBank: 112422..113942
CDS feature translation: ['VYLVFTTNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRINMELLPHAHSIFAPWLMIVGTIQIIYAASTSPGQRNLKKRIAYSSVSHMGFILIGIASITDTGLNGAILQIISHGFIGAALFFLAGTSYDRIRLVYLDQMGGVAIPMPKIFTMFSSFSMASLALPGMSGFVAEVMVFLGIITSQKYLLIPKIAITFVMAIGMILTPIYSLSMLRQMFYGYKLFNTPNSYVFDSGPRELFVSISIFLPVIGIGMYPDFVLSLSVDKVEGILSNYFYR']
______________________________________________
GenBank file: plastomes/table2asn/cp/01_prot_ndhD/cp_ndhD-coord-fixed.gb
Translated sequence first symbols: TNKFPWLTIIVVLPIFAGSFI

gene strand: (-)
Original 

In [7]:
from Bio import SeqIO

input_gb = "plastomes/table2asn/cp/01_prot_ndhD/cp_ndhD-coord-fixed.gb"      # Define file directly by relative path
outdir=f"{workdir}/02_start_codon"
output_gb = f"{outdir}/cp_start_codons_fixed.gb"
target_genes = {"psbL", "ndhD"}

os.makedirs(outdir, exist_ok=True)

records = list(SeqIO.parse(input_gb, "gb"))

for record in records:
    for feature in record.features:
        if feature.type == "CDS" and feature.qualifiers.get("gene", [""])[0] in target_genes:
            gene = feature.qualifiers["gene"][0]
            loc = feature.location
            
            # 1. Extract genomic start codon & determine 1-based coordinates
            if loc.strand == -1:
                # Reverse strand: start codon is at the 3' end of the genomic interval
                genomic_codon = record.seq[loc.end-3:loc.end].reverse_complement()
                start_codon_pos = f"complement({loc.end-2}..{loc.end})"
            else:
                # Forward strand: start codon is at the 5' end
                genomic_codon = record.seq[loc.start:loc.start+3]
                start_codon_pos = f"{loc.start+1}..{loc.start+3}"
                
            genomic_codon_str = str(genomic_codon).upper()
            
            # Skip if already standard ATG
            if genomic_codon_str == "ATG":
                print(f"{gene}: Already has standard ATG start. No change needed.")
                continue
                
            # 2. Update /translation to start with M (edited protein)
            if "translation" in feature.qualifiers:
                current_trans = feature.qualifiers["translation"][0]
                if current_trans[0] != "M":
                    feature.qualifiers["translation"][0] = "M" + current_trans[1:]
                    print(f"{gene}: Updated translation to start with M.")
                    
            # 3. Add /transl_except qualifier (NCBI official format)
            except_val = f"(pos:{start_codon_pos},aa=M)"
            feature.qualifiers["transl_except"] = [except_val]
            print(f"{gene}: Added /transl_except=\"{except_val}\"")
            
            # 4. Add clarifying note (optional but recommended for reviewers)
            if "note" not in feature.qualifiers:
                feature.qualifiers["note"] = []
            feature.qualifiers["note"].append("Start codon created by RNA editing (C-to-U)")

SeqIO.write(records, output_gb, "gb")
print(f"\nSaved corrected GenBank to: {output_gb}")

psbL: Updated translation to start with M.
psbL: Added /transl_except="(pos:complement(62023..62025),aa=M)"
ndhD: Updated translation to start with M.
ndhD: Added /transl_except="(pos:complement(113922..113924),aa=M)"

Saved corrected GenBank to: plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb


In [8]:
# Check CDS for start codon exception
from Bio import SeqIO

gb_file = "plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb" # file with fixed tRNAs
genes = ["psbL", "ndhD"]

with open(gb_file) as handle:
    for record in SeqIO.parse(handle, "gb"):
        for feature in record.features:
            if feature.qualifiers.get('gene', "[]")[0] in genes:
                if feature.type in ["CDS"]:
                    print(feature)

type: CDS
location: [61908:62025](-)
qualifiers:
    Key: gene, Value: ['psbL']
    Key: note, Value: ['Start codon created by RNA editing (C-to-U)']
    Key: product, Value: ['photosystem II subunit L']
    Key: transl_except, Value: ['(pos:complement(62023..62025),aa=M)']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MTQSNPNEQNVELNRTSLYWGLLLIFVLAVLFSNYFFN']

type: CDS
location: [112421:113924](-)
qualifiers:
    Key: gene, Value: ['ndhD']
    Key: note, Value: ['Start codon created by RNA editing (C-to-U)']
    Key: product, Value: ['NADH dehydrogenase subunit D']
    Key: transl_except, Value: ['(pos:complement(113922..113924),aa=M)']
    Key: transl_table, Value: ['11']
    Key: translation, Value: ['MNKFPWLTIIVVLPIFAGSFILFFPHKGNRVIRWYTICICMLELILTTYAFCYHFQLDDPLIQLVEDYKWINFFDFRWKLGIDGLSIGPVLLTGFITTLATLAAWPVTRDSRLFHFLMLAMYSGQIGSFSSRDLLLFFIMWELELIPVYLLLSMWGGKKRLYSATKFILYTAGGSIFLLMGVLGVGLYGSNEPTLNFETSVNQSYPVALEIIFYIGFFIAFAVKSPILPLHTWLPDTHGEAHYSTCMLLAGILLKMGAYGLIRI

### Compare features between cc and cp annotations
Prepare pandas dataframe that harbours all feature data

In [9]:
import pandas as pd
import numpy as np
from Bio import SeqRecord, SeqIO, SeqFeature

# variables
c_pur = "plastomes/table2asn/cp/02_start_codon/cp_start_codons_fixed.gb"
c_cal = "plastomes/table2asn/cc/fixed_table2asn/output.gbf"

regions = {}
dataset = []

# Do initial parsing
for acc in c_cal, c_pur:
    with open(acc, "r") as handle:
        for record in SeqIO.parse(handle, "genbank"):
            features = record.features
            # check if there features other than 'source'
            if len(features) > 1:
                for feature in features:
                    # check for genome regions
                    if feature.type in ['misc_feature', 'repeat_region']:
                        label = feature.qualifiers.get('note', "")[0].split()[-1].strip("()")
                        regions[label] = [int(feature.location.start), int(feature.location.end)]
                # filling the dataframe
                if acc == c_cal:
                    organism = "Crepis callicephala"
                else:
                    organism = record.annotations.get('organism', "")
                for feature in features:
                    feature_type = feature.type
                    if feature_type in ['gene', 'CDS', 'tRNA', 'rRNA']:
                        feature_dict = {}
                        # gene and start position need for further check
                        gene = feature.qualifiers.get('gene', "")[0]
                        start = int(feature.location.start)
                        end = int(feature.location.end)
                        if feature_type == 'gene':
                            feature_dict['organism'] = organism
                            feature_dict['gene'] = gene
                            feature_dict['start'] = start
                            feature_dict['end'] = end
                            feature_dict['gene_len'] = int(len(feature))
                            feature_dict['spanned'] = False
                            if feature.location.strand:
                                feature_dict['strand'] = int(feature.location.strand)

                            for r_name, r_range in regions.items():
                                if r_range[0] <= start < r_range[1]:
                                    feature_dict['region'] = r_name
                                    feature_dict['spanned'] = (end >= r_range[1])
                                    break

                            dataset.append(feature_dict)

                        if feature_type in ['CDS', 'tRNA', 'rRNA']:
                            for feat in dataset[-4:]:
                                if gene == feat['gene'] and start == feat['start']:
                                    feat['type'] = feature_type
                                    feat['feature_len'] = len(feature)
                                    sequence = feature.extract(record.seq)
                                    feat['na_seq'] = f"{sequence[:6]}...{sequence[-6:]}"
                                    if feature_type == 'CDS':
                                        aa_seq = feature.qualifiers.get('translation', [""])[0]
                                        feat['aa_seq'] = aa_seq[:6]
                                    break  # dataset is already updated here
                                    

df = pd.DataFrame(dataset)
#print(df.head())
display(df)

,organism,gene,start,end,gene_len,spanned,strand,region,type,feature_len,na_seq,aa_seq
0,Crepis callicephala,rps12,67368,95793,907,True,-1.0,LSC,CDS,372.0,ATGCCA...AAATAA,MPTIKQ
1,Crepis callicephala,trnH-GUG,2,77,75,False,-1.0,LSC,tRNA,75.0,GCGGAT...TCGCCC,NaN
2,Crepis callicephala,psbA,474,1536,1062,False,-1.0,LSC,CDS,1062.0,ATGACT...GGATAA,MTAILE
3,Crepis callicephala,trnK-UUU,1758,4358,2600,False,-1.0,LSC,tRNA,74.0,TGGGTT...ACCCAT,NaN
4,Crepis callicephala,matK,2079,3591,1512,False,-1.0,LSC,CDS,1512.0,ATGGAG...GAATGA,MEKFQS
...,...,...,...,...,...,...,...,...,...,...,...,...
266,Crepis purpurea,ycf2,140899,147754,6855,False,-1.0,IRA,CDS,6855.0,ATGACA...CCATGA,MTGHEF
267,Crepis purpurea,trnM-CAU,147865,147940,75,False,1.0,IRA,tRNA,75.0,GCATCC...ATGCAC,NaN
268,Crepis purpurea,rpl23,148104,148386,282,False,1.0,IRA,CDS,282.0,ATGGAT...ACTTAA,MDGIRY
269,Crepis purpurea,rpl2,148404,149894,1490,False,1.0,IRA,CDS,825.0,ATGGCG...AAATAG,MAIHLY


In [10]:
df_CDS = df[df['type'] == 'CDS']
df_CDS 
#display(df_CDS)

pivot_CDS_len = df_CDS.pivot_table(
    values='feature_len',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_CDS_len.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

organism,Crepis callicephala,Crepis purpurea
gene,,
petD,525.0,483.0
psaA,2253.0,1776.0
rpl22,465.0,540.0
rpoC1,2076.0,1791.0
rpoC2,4125.0,2274.0


### Investigate start codones among annotations



In [11]:
pivot_CDS_aa = df_CDS.pivot_table(
    values='aa_seq',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_CDS_aa.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

organism,Crepis callicephala,Crepis purpurea
gene,,
petD,MSGSYG,MGVTKK
rps19,MTRSLK,VTRSLK


In [12]:
pivot_CDS_na = df_CDS.pivot_table(
    values='na_seq',
    index='gene',
    columns=['organism'],
    aggfunc='first'
)

piv = pivot_CDS_na.sort_index(axis=1)

# Keep only rows where values differ (excluding NaN mismatches)
diff_mask = (piv['Crepis callicephala'] != piv['Crepis purpurea']) #& \
#            piv['Crepis callicephala'].notna() & \
#            piv['Crepis purpurea'].notna()

diff_genes = piv[diff_mask]
display(diff_genes)

organism,Crepis callicephala,Crepis purpurea
gene,,
petD,ATGTCC...TTTTAA,ATGGGA...TTTTAA
psaA,ATGATT...GGATAA,ATGATT...TCTTAG
rpl22,ATGTTA...AAATAA,ATGTTA...TTCTGA
rpoC1,ATGATC...GGTTAA,ATGATC...ATTTAG
rpoC2,ATGGCA...TTTTAG,ATGGCA...AATTGA


In [13]:
issued_genes = ['rps19', 'petD', 'psaA', 'rpl22', 'rpoC1', 'rpoC2']

df_lookup = df[df['gene'].isin(issued_genes)].copy()
# Pivot with multiple value columns
pivot_multi = df_lookup.pivot_table(
    index='gene',
    columns='organism',
    values=['start', 'end', 'feature_len', 'na_seq', 'aa_seq', 'strand'],
    aggfunc='first'  # or 'first' since each gene/organism combo should be unique
)

# Optional: Flatten column MultiIndex for easier reading
pivot_multi.columns = ['_'.join(col).strip() for col in pivot_multi.columns.values]

display(pivot_multi.head())

,aa_seq_Crepis callicephala,aa_seq_Crepis purpurea,end_Crepis callicephala,end_Crepis purpurea,feature_len_Crepis callicephala,feature_len_Crepis purpurea,na_seq_Crepis callicephala,na_seq_Crepis purpurea,start_Crepis callicephala,start_Crepis purpurea,strand_Crepis callicephala,strand_Crepis purpurea
gene,,,,,,,,,,,,
petD,MSGSYG,MGVTKK,75361,75387,525.0,483.0,ATGTCC...TTTTAA,ATGGGA...TTTTAA,74836,74204,1.0,1.0
psaA,MIIRSP,MIIRSP,41258,41293,2253.0,1776.0,ATGATT...GGATAA,ATGATT...TCTTAG,39005,39517,-1.0,-1.0
rpl22,MLNKRT,MLNKRT,81529,81543,465.0,540.0,ATGTTA...AAATAA,ATGTTA...TTCTGA,81064,81003,-1.0,-1.0
rpoC1,MIDRYT,MIDRYT,18963,18702,2076.0,1791.0,ATGATC...GGTTAA,ATGATC...ATTTAG,16164,16182,1.0,1.0
rpoC2,MAERPT,MAERPT,23201,21367,4125.0,2274.0,ATGGCA...TTTTAG,ATGGCA...AATTGA,19076,19093,1.0,1.0


In [14]:
# comparing translations, first cc, then cp

for issued in issued_genes:
    print(f"\n{issued}")
    for gb in c_cal, c_pur:
        records = SeqIO.parse(gb, "gb")
        for record in records:
            for feature in record.features:
                if feature.type == 'CDS':
                    gene = feature.qualifiers['gene'][0]
                    if gene == issued:
                        cds = feature.extract(record.seq)
                        aa = cds.translate(to_stop=False)
                        print(f"{aa}")


rps19
VTRSLKKNPFVANNLLKKINKLNTKAEKEIIITWSRASTIIPIMVGHTIAIHNGKEHLPIYITDRMVGHKLGEFAPTLNFRGHAKSDNRSRR*
VTRSLKKNPFVANNLLKKINKLNTKAEKEIIITWSRASTIIPIMVGHTIAIHNGKEHLPIYITDRMVGHKLGEFAPTLNFRGHAKSDNRSRR*

petD
MSGSYGGWIYKNSPIPITKKPDLNDPVLRAKLAKGMGHNYYGEPAWPNDLLYIFPVVILGTIACNVGLAVLEPSMIGEPADPFATPLEILPEWYFFPVFQILRTVPNKLLGVLLMVSVPAGLLTVPFLENVNKFQNPFRRPVATTVFLIGTAVALWLGIGATLPIDKSLTLGLF*
MGVTKKPDLNDPVLRAKLAKGMGHNYYGEPAWPNDLLYIFPVVILGTIACNVGLAVLEPSMIGEPADPFATPLEILPEWYFFPVFQILRTVPNKLLGVLLMVSVPAGLLTVPFLENVNKFQNPFRRPVATTVFLIGTAVALWLGIGATLPIDKSLTLGLF*

psaA
MIIRSPEPEVKILVDRDHIKTSFEEWARPGHFSRTIAKGPETTTWIWNLHADAHDFDSHTSDLEEISRKVFSAHFGQLSIIFLWLSGMYFHGARFSNYEAWLSDPTHIGPSAQVVWPIVGQEILNGDVGGGFRGIQITSGFFQIWRASGITSELQLYCTAIGALVFAALMLFAGWFHYHKAAPKLAWFQDVESMLNHHLAGLLGLGSLSWAGHQVHVSLPINQFLNAGVDPKEIPLPHEFILNRDLLAQLYPSFAEGATPFFTLNWSKYADFLTFRGGLDPVTGGLWLTDTAHHHLAIAILFLIAGHMYRTNWGIGHGLKDILEAHKGPFTGQGHKGLYEILTTSWHAQLSLNLAMLGSLTIVVAHHMYAMPPYPYLATDYGTVLSLFTHHMWIGGFLIVGAAAHAAIFMVRDYDPTTRYNDLLDRVLRHRDAIISHLNWACIFLGFHSFGLYIH

Save reference sequences for issued genes

In [ ]:
import os
import time
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

Entrez.email = "asantaraqtasli@gmail.com"

accessions = {
    "NC_007578.1": "Lactuca sativa",
    "NC_016893.1": "Lapsana communis",
    "NC_046516.1": "Youngia japonica",
    "NC_031396.1": "Taraxacum mongolicum",
    "NC_061360.1": "Crepis rigescens",
}
out_dir = "plastomes/refs"
os.makedirs(out_dir, exist_ok=True)

for acc, species in accessions.items():
    tag = species.lower().replace(" ", "_")
    print(f"Fetching {species} ({acc})...")
    
    with Entrez.efetch(db="nucleotide", id=acc, rettype="gb", retmode="text") as handle:
        rec = SeqIO.read(handle, "gb")
        
    for gene in issued_genes:
        cds_feat, gene_feat = None, None
        for feat in rec.features:
            g = feat.qualifiers.get("gene", [""])[0]
            if g == gene:
                if feat.type == "CDS":
                    cds_feat = feat
                elif feat.type == "gene":
                    gene_feat = feat
                    
        if not cds_feat:
            print(f"  Skipping {gene}: CDS not found.")
            continue
            
        cds_seq = cds_feat.extract(rec.seq)
        genomic_seq = gene_feat.extract(rec.seq) if gene_feat else cds_feat.extract(rec.seq)
        prot_seq = Seq(cds_feat.qualifiers.get("translation", [str(cds_seq.translate())])[0])
        
        prefix = f"{gene}_{tag}"
        seq_map = {
            "genomic.fna": genomic_seq,
            "cds.fna": cds_seq,
            "protein.faa": prot_seq
        }
        
        for ext, seq in seq_map.items():
            filepath = os.path.join(out_dir, f"{prefix}.{ext}")
            SeqIO.write(SeqRecord(seq, id=prefix), filepath, "fasta")
            
        print(f"  Saved {prefix}.*")
        time.sleep(0.5)

Fetching Lactuca sativa (NC_007578.1)...
  Saved rps19_lactuca_sativa.*
  Saved petD_lactuca_sativa.*
  Saved psaA_lactuca_sativa.*
  Saved rpl22_lactuca_sativa.*
  Saved rpoC1_lactuca_sativa.*
  Saved rpoC2_lactuca_sativa.*
Fetching Lapsana communis (NC_016893.1)...
  Skipping rps19: CDS not found.
  Skipping petD: CDS not found.
  Skipping psaA: CDS not found.
  Skipping rpl22: CDS not found.
  Skipping rpoC1: CDS not found.
  Skipping rpoC2: CDS not found.


In [112]:
def build_cds_index(filepath):
    idx = {}
    for rec in SeqIO.parse(filepath, "genbank"):
        for feat in rec.features:
            if feat.type == "CDS":
                gene_list = feat.qualifiers.get("gene", [])
                if gene_list:
                    gene = gene_list[0]
                    dna = feat.extract(rec.seq)
                    # Translate the extracted sequence, not the feature object
                    aa = dna.translate(table=11)
                    if str(aa).endswith("*"):
                        aa = aa[:-1]
                    idx[gene] = {"dna": str(dna), "aa": str(aa)}
    return idx

# 1. Parse files ONCE
cal_idx = build_cds_index(c_cal)
pur_idx = build_cds_index(c_pur)

def print_seq_diff(seq1, seq2, seq_type="Seq"):
    if seq1 == seq2:
        print(f"  [OK] {seq_type} identical ({len(seq1)} chars)")
        return
    
    print(f"  [DIFF] {seq_type} differs ({len(seq1)} vs {len(seq2)} chars)")
    for i, (a, b) in enumerate(zip(seq1, seq2)):
        if a != b:
            print(f"  First mismatch at pos {i}: '{a}' -> '{b}'")
            s, e = max(0, i-5), min(len(seq1), len(seq2), i+6)
            print(f"  Context: ...{seq1[s:e]}... ...{seq2[s:e]}...")
            break
    if len(seq1) != len(seq2):
        print(f"  Length difference: {abs(len(seq1)-len(seq2))} bp/aa indel detected")

# 2. Compare issued genes
for gene in issued_genes:
    print(f"\n{'='*45}")
    print(f"Gene: {gene}")
    
    c1, c2 = cal_idx.get(gene), pur_idx.get(gene)
    if not c1 or not c2:
        print("Missing CDS in one organism. Skipping.")
        continue
        
    print_seq_diff(c1["dna"], c2["dna"], "DNA")
    print_seq_diff(c1["aa"], c2["aa"], "Protein")


Gene: rps19
  [OK] DNA identical (279 chars)
  [OK] Protein identical (92 chars)

Gene: petD
  [DIFF] DNA differs (525 vs 483 chars)
  First mismatch at pos 3: 'T' -> 'G'
  Context: ...ATGTCCGGT... ...ATGGGAGTA...
  Length difference: 42 bp/aa indel detected
  [DIFF] Protein differs (174 vs 160 chars)
  First mismatch at pos 1: 'S' -> 'G'
  Context: ...MSGSYGG... ...MGVTKKP...
  Length difference: 14 bp/aa indel detected

Gene: psaA
  [DIFF] DNA differs (2253 vs 1776 chars)
  First mismatch at pos 1739: 'G' -> 'A'
  Context: ...GGGGGGACATG... ...GGGGGACATGT...
  Length difference: 477 bp/aa indel detected
  [DIFF] Protein differs (750 vs 591 chars)
  First mismatch at pos 580: 'T' -> 'H'
  Context: ...PGRGGTCQVSA... ...PGRGGHVKYPL...
  Length difference: 159 bp/aa indel detected

Gene: rpl22
  [DIFF] DNA differs (465 vs 540 chars)
  First mismatch at pos 404: 'A' -> 'C'
  Context: ...AAAAAACATAC... ...AAAAACATACA...
  Length difference: 75 bp/aa indel detected
  [DIFF] Protein differs

### Compare genes among repeat regions

In [ ]:
df_repeats = df[df['region'].isin(['IRA', 'IRB'])]

pivot_table = df_repeats.pivot_table(
    values='feature_len',
    index='gene',
    columns=['organism', 'region'],
    aggfunc='first'
)

# Reorder levels in the columns
#pivot_table = df2.reorder_levels([1, 0], axis=1)

# Sort the columns
pivot_table = pivot_table.sort_index(axis=1)

display(pivot_table)


organism Crepis callicephala         Crepis purpurea        
region                   IRA     IRB             IRA     IRB
gene                                                        
ndhB                  1533.0  1533.0          1533.0  1533.0
ndhF                     NaN  2226.0             NaN  2226.0
rpl2                   825.0   825.0           825.0   825.0
rpl23                  282.0   282.0           282.0   282.0
rps7                   468.0   468.0           468.0   468.0
rrn16                 1490.0  1490.0          1490.0  1490.0
rrn23                 2810.0  2810.0          2810.0  2810.0
rrn4.5                 103.0   103.0           103.0   103.0
rrn5                   122.0   116.0           121.0   122.0
trnA-UGC                73.0    73.0            73.0    73.0
trnI-GAU                73.0    73.0             NaN     NaN
trnL-CAA                81.0    81.0            83.0    83.0
trnM-CAU                 NaN     NaN            75.0    75.0
trnN-GUU                72.0    72.0            74.0    74.0
trnR-ACG                74.0    74.0            75.0    75.0
trnV-GAC                73.0    72.0            72.0    72.0
ycf15                  192.0   192.0           192.0   192.0
ycf2                  6855.0  6855.0          6855.0  6855.0